# 模块(二): 匹配对照组 — 放宽版

**日期:** 2026-05-12
**改进:** NumA/NumR 从 Exact Match 改为 +-10% 容差
**策略:** DuckDB 大表 join，只读需要的列

In [ ]:
import duckdb
import pandas as pd
import time
import os

con = duckdb.connect()
con.execute("SET memory_limit = '150GB'")
print("DuckDB 已连接, 内存上限 150G")

## Step 1: 加载 Focal Paper 列表

从 df_containretra3.csv 读取 275,406 篇 focal paper

In [ ]:
focal_path = '/Data4/yutao_wen/processed/df_containretra3.csv'
con.execute(f"CREATE OR REPLACE TABLE focals AS SELECT * FROM read_csv_auto('{focal_path}')")

n_focals = con.execute('SELECT COUNT(*) FROM focals').fetchone()[0]
print(f'Focal papers 加载完成: {n_focals:,} 篇')
print(f'列: id, reference_ids, Year, Journal_id, RPYear, RYear, RJournal_id, Num_Retra, is_self_retracted')

## Step 2: 从 Papers.csv 获取 Focal 自己的 NumA/NumR

我们已有 Year 和 Journal_id, 还需要 NumA(num_authors) 和 NumR(num_references)

In [ ]:
papers_path = '/data6/Data1/DATA/Dimensions2024/20240101/Papers.csv'

print('正在从 Papers.csv 提取 focals 的 NumA 和 NumR...')
print('  只查 275K 个 ID, 预计 2-3 分钟')
t0 = time.time()

# 注册 focal ID 列表
con.execute("CREATE OR REPLACE TEMP TABLE focal_ids AS SELECT id FROM focals")

# DuckDB 只读需要的行和列
con.execute(f'''
    CREATE OR REPLACE TEMP TABLE focal_covariates AS
    SELECT p.id, p.num_authors AS NumA, p.num_references AS NumR
    FROM read_csv_auto('{papers_path}',
        columns={{'id': 'VARCHAR', 'num_authors': 'BIGINT', 'num_references': 'BIGINT'}}) AS p
    WHERE p.id IN (SELECT id FROM focal_ids)
''')

matched = con.execute('SELECT COUNT(*) FROM focal_covariates WHERE NumA IS NOT NULL').fetchone()[0]
print(f'耗时: {(time.time()-t0)/60:.1f} 分钟')
print(f'匹配到 NumA/NumR 的 focal: {matched:,} ({matched/n_focals*100:.1f}%)')
print(f'缺失: {n_focals-matched:,}')

# 合并回 focals 表
con.execute("DROP TABLE IF EXISTS focals_full")
con.execute('''
    CREATE TABLE focals_full AS
    SELECT f.*, c.NumA, c.NumR
    FROM focals f
    LEFT JOIN focal_covariates c ON f.id = c.id
    WHERE c.NumA IS NOT NULL AND c.NumR IS NOT NULL
''')
n_valid = con.execute('SELECT COUNT(*) FROM focals_full').fetchone()[0]
print(f'有效 focal (NumA/NumR 完整): {n_valid:,}')

## Step 3: 加载全量 Papers 的 Year/Journal/NumA/NumR

从 33G Papers.csv 提取所有论文的这四个维度, 用于匹配

In [ ]:
print('正在加载全量 Papers 的四维度数据 (33G)...')
print('  预计 5-8 分钟, 仅读取 4 列')
t0 = time.time()

con.execute(f'''
    CREATE OR REPLACE TABLE all_papers AS
    SELECT p.id,
           CAST(substr(p.date_normal, 1, 4) AS INTEGER) AS Year,
           p."journal.id" AS Journal_id,
           p.num_authors AS NumA,
           p.num_references AS NumR
    FROM read_csv_auto('{papers_path}',
        types={{'date_normal': 'VARCHAR'}}) AS p
    WHERE p.date_normal IS NOT NULL AND p.date_normal != ''
      AND p."journal.id" IS NOT NULL
      AND p.num_authors IS NOT NULL
      AND p.num_references IS NOT NULL
''')

n_all = con.execute('SELECT COUNT(*) FROM all_papers').fetchone()[0]
print(f'耗时: {(time.time()-t0)/60:.1f} 分钟')
print(f'有效论文: {n_all:,} 篇')
print(f'Focal 占比: {n_valid/n_all*100:.2f}%')

## Step 4: 匹配 — 同期刊 + 同年 + NumA/NumR 10% 容差

对每篇 focal, 在 all_papers 中找:
- 同一个 journal_id
- 同一个 Year
- |NumA_diff| <= focal.NumA * 10%
- |NumR_diff| <= focal.NumR * 10%
- 排除 focal 自己

In [ ]:
A_TOL = 0.10  # NumA 容差 (10%)
R_TOL = 0.10  # NumR 容差 (10%)

print(f'容差: NumA +-{A_TOL*100:.0f}%, NumR +-{R_TOL*100:.0f}%')
print('正在执行四维度匹配 (同期刊同年 + 容差)...')
t0 = time.time()

con.execute(f'''
    CREATE OR REPLACE TABLE df_match AS
    SELECT
        f.id AS exp_id,
        f.Year AS PYear,
        f.Journal_id,
        f.NumA AS f_NumA,
        f.NumR AS f_NumR,
        c.id AS ctrl_id,
        c.NumA AS c_NumA,
        c.NumR AS c_NumR
    FROM focals_full f
    INNER JOIN all_papers c
        ON f.Journal_id = c.Journal_id
       AND f.Year = c.Year
    WHERE c.id != f.id
      AND ABS(f.NumA - c.NumA) <= f.NumA * {A_TOL}
      AND ABS(f.NumR - c.NumR) <= f.NumR * {R_TOL}
''')

n_pairs = con.execute('SELECT COUNT(*) FROM df_match').fetchone()[0]
n_focal_matched = con.execute('SELECT COUNT(DISTINCT exp_id) FROM df_match').fetchone()[0]
n_ctrl_unique = con.execute('SELECT COUNT(DISTINCT ctrl_id) FROM df_match').fetchone()[0]

elapsed = time.time() - t0
print(f'\n匹配完成! 耗时: {elapsed/60:.1f} 分钟')
print(f'匹配对数: {n_pairs:,}')
print(f'有至少 1 个候选控制的 focal: {n_focal_matched:,} ({n_focal_matched/n_valid*100:.1f}%)')
print(f'无候选控制的 focal: {n_valid-n_focal_matched:,} ({(n_valid-n_focal_matched)/n_valid*100:.1f}%)')
print(f'唯一控制论文: {n_ctrl_unique:,}')
print(f'平均每个 focal 匹配数: {n_pairs/n_focal_matched:.1f}')

## Step 5: 查看每篇 Focal 匹配了多少候选控制

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 每篇 focal 的匹配数分布
ctrl_counts = con.execute('''
    SELECT exp_id, COUNT(*) AS n_controls
    FROM df_match
    GROUP BY exp_id
''').df()

print(f'每篇 Focal 的候选控制数分布:')
print(f'  均值: {ctrl_counts["n_controls"].mean():.1f}')
print(f'  中位数: {ctrl_counts["n_controls"].median():.0f}')
print(f'  最小值: {ctrl_counts["n_controls"].min()}')
print(f'  最大值: {ctrl_counts["n_controls"].max()}')
for p in [25, 50, 75, 90, 95, 99]:
    print(f'  P{p}: {ctrl_counts["n_controls"].quantile(p/100):.0f}')

# 直方图
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(ctrl_counts['n_controls'], bins=50, color='#2E86AB', edgecolor='white', alpha=0.8)
ax.axvline(ctrl_counts['n_controls'].median(), color='#E94D3C', linestyle='--', label=f'Median={ctrl_counts["n_controls"].median():.0f}')
ax.set_xlabel('Number of Candidate Controls per Focal', fontsize=12)
ax.set_ylabel('Number of Focal Papers', fontsize=12)
ax.set_title(f'Distribution of 1:N Matching (N={len(ctrl_counts):,} focals)', fontsize=14, weight='bold')
ax.legend()
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('/Data4/yutao_wen/processed/Control_Match_Distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'图片已保存')

## Step 6: 保存匹配结果

In [ ]:
output_path = '/Data4/yutao_wen/processed/df_match.csv'
con.execute(f"COPY df_match TO '{output_path}' (HEADER, DELIMITER ',')")
size_mb = os.path.getsize(output_path) / 1024 / 1024

print(f'已保存: {output_path}')
print(f'大小: {size_mb:.1f} MB')
print(f'行数: {n_pairs:,}')
print(f'有候选的 focal: {n_focal_matched:,}')
print(f'\n模块(二) 完成! 下一步: 模块(三) 事前事后引用计算')
print(f'\n容差参数: NumA +-{A_TOL*100:.0f}%, NumR +-{R_TOL*100:.0f}%')
print(f'容差等级: 宽松匹配 (非 Exact)')